# 01 — Data Loading: CDC SVI + FEMA County Matrix

## What this notebook does
Builds the input dataset for the County-Level Triage Engine by joining two sources:

1. **CDC Social Vulnerability Index (SVI) 2018** — county-level composite vulnerability score
   (0 = least vulnerable, 1 = most vulnerable) plus population and land area
2. **FEMA PA project data** — used to build a county × disaster matrix and compute
   prior disaster exposure per county

## Why these two sources?
The SVI captures *structural* vulnerability (poverty, housing, minority status, language barriers,
vehicle access) — factors that determine how hard a county is hit and how slowly it recovers.
Population density captures *scale of exposure* — the same storm hits more people in a dense county.
Prior disaster exposure captures *repeated burden* — counties that appear repeatedly in FEMA
declarations have less time to recover between events.

Together these three signals let FEMA triage which counties need immediate technical assistance
deployment vs which can follow standard review timelines.

## Unit of analysis
One row per **county × disaster declaration**. Each row gets a Priority Score (1/2/3).

In [1]:
import pandas as pd
import numpy as np
import requests
import os
import warnings
warnings.filterwarnings('ignore')

RAW       = '../data/raw/'
PROCESSED = '../data/processed/'
PHASE1_RAW = '../../data/raw/'

os.makedirs(RAW, exist_ok=True)
os.makedirs(PROCESSED, exist_ok=True)

print('Libraries loaded.')
print(f'Phase 1 raw data path: {os.path.abspath(PHASE1_RAW)}')

Libraries loaded.
Phase 1 raw data path: c:\Users\63429\Desktop\AI2\GROUPWORK\project\data\raw


## Step 1 — Download CDC Social Vulnerability Index

The CDC SVI is published every 2 years. We use the **2018 edition** (most recent that fully
overlaps with our FEMA training period). It is a free public dataset from the CDC/ATSDR
and requires no API key.

Key columns we use:
- `FIPS`: 5-digit county FIPS code (join key)
- `RPL_THEMES`: overall SVI percentile (0–1), our vulnerability score
- `E_TOTPOP`: total population estimate
- `AREA_SQMI`: county land area in square miles → used to compute population density
- `STATE`, `COUNTY`: for display purposes

Values of -999 in the SVI indicate missing data and are replaced with NaN.

In [4]:
SVI_PATH = RAW + 'SVI2018_US_COUNTY.csv'

if not os.path.exists(SVI_PATH):
    print('Downloading CDC SVI 2018 county-level data...')
    urls = [
        'https://svi.cdc.gov/Documents/Data/2018_SVI_Data/CSV/SVI2018_US_COUNTY.csv',
        'https://www.atsdr.cdc.gov/placeandhealth/svi/documentation/csv/SVI2018_US_COUNTY.csv',
    ]
    downloaded = False
    for url in urls:
        try:
            r = requests.get(url, timeout=60)
            if r.status_code == 200:
                with open(SVI_PATH, 'wb') as f:
                    f.write(r.content)
                print(f'  Saved to {SVI_PATH}')
                downloaded = True
                break
        except Exception as e:
            print(f'  URL failed: {url} — {e}')
    if not downloaded:
        print('\nAuto-download failed. Please manually download from:')
        print('https://www.atsdr.cdc.gov/placeandhealth/svi/data_documentation_download.html')
        print(f'Save as: {os.path.abspath(SVI_PATH)}')
else:
    print(f'SVI file already exists: {SVI_PATH}')

svi_raw = pd.read_csv(SVI_PATH, low_memory=False)
print(f'\nSVI shape: {svi_raw.shape}')
print(f'Columns sample: {list(svi_raw.columns[:10])}')

SVI file already exists: ../data/raw/SVI2018_US_COUNTY.csv

SVI shape: (3142, 123)
Columns sample: ['ST', 'STATE', 'ST_ABBR', 'COUNTY', 'FIPS', 'LOCATION', 'AREA_SQMI', 'E_TOTPOP', 'M_TOTPOP', 'E_HU']


In [5]:
# Select and clean SVI columns
SVI_COLS = ['FIPS', 'STATE', 'COUNTY', 'RPL_THEMES', 'E_TOTPOP', 'AREA_SQMI']
# Handle column name variations across SVI editions
col_map = {}
for col in SVI_COLS:
    matches = [c for c in svi_raw.columns if c.upper() == col.upper()]
    if matches:
        col_map[matches[0]] = col

svi = svi_raw[list(col_map.keys())].rename(columns=col_map).copy()

# Replace SVI sentinel value -999 with NaN
svi = svi.replace(-999, np.nan)

# Standardise FIPS to 5-digit string
svi['FIPS'] = svi['FIPS'].astype(str).str.zfill(5)

# Compute population density
svi['pop_density'] = svi['E_TOTPOP'] / svi['AREA_SQMI'].replace(0, np.nan)

# Rename for clarity
svi = svi.rename(columns={
    'RPL_THEMES': 'svi_score',
    'E_TOTPOP':   'population',
    'AREA_SQMI':  'area_sqmi',
})

# Drop rows with missing SVI score
svi = svi.dropna(subset=['svi_score'])

print(f'Clean SVI shape : {svi.shape}')
print(f'SVI score range : {svi["svi_score"].min():.3f} – {svi["svi_score"].max():.3f}')
print(f'Pop density range: {svi["pop_density"].min():.1f} – {svi["pop_density"].max():.1f} people/sqmi')
print(svi.head(3))

Clean SVI shape : (3141, 7)
SVI score range : 0.000 – 1.000
Pop density range: 0.0 – 72053.0 people/sqmi
    FIPS    STATE   COUNTY  svi_score  population   area_sqmi  pop_density
1  01001  ALABAMA  Autauga     0.4354       55200  594.443459    92.859967
2  01009  ALABAMA   Blount     0.4242       57645  644.830460    89.395591
3  01013  ALABAMA   Butler     0.8653       20025  776.838201    25.777569


## Step 2 — Build FEMA County × Disaster Matrix

The raw PA projects file contains one row per project, with the county FIPS and
disaster number on each row. We extract the unique county × disaster combinations
to build our triage matrix.

We also compute **prior_disasters_5yr** per county — the number of distinct disaster
declarations that included this county in the 5 years before the current declaration.
Counties with high prior exposure are under cumulative stress and need faster support.

In [6]:
# Find FEMA PA projects file in Phase 1 raw folder
import glob

pa_candidates = (
    glob.glob(PHASE1_RAW + '*project*.csv') +
    glob.glob(PHASE1_RAW + '*Project*.csv') +
    glob.glob(PHASE1_RAW + '*public*.csv') +
    glob.glob(PHASE1_RAW + '*.csv')
)

print('Candidate files:')
for f in pa_candidates[:5]:
    print(f'  {f}')

PA_PATH = pa_candidates[0] if pa_candidates else None
if PA_PATH is None:
    raise FileNotFoundError('No FEMA PA CSV found in Phase 1 raw folder. Check path.')

pa = pd.read_csv(PA_PATH, low_memory=False)
print(f'\nPA projects shape: {pa.shape}')
print(f'Columns: {list(pa.columns)}')

Candidate files:
  ../../data/raw\pa_funded_projects.csv
  ../../data/raw\pa_funded_projects.csv
  ../../data/raw\disaster_declarations.csv
  ../../data/raw\median_household_income.csv
  ../../data/raw\national_risk_index.csv

PA projects shape: (810656, 25)
Columns: ['disasterNumber', 'declarationDate', 'incidentType', 'pwNumber', 'applicationTitle', 'applicantId', 'damageCategoryCode', 'damageCategoryDescrip', 'projectStatus', 'projectProcessStep', 'projectSize', 'county', 'countyCode', 'stateAbbreviation', 'stateNumberCode', 'projectAmount', 'federalShareObligated', 'totalObligated', 'lastObligationDate', 'firstObligationDate', 'mitigationAmount', 'gmProjectId', 'gmApplicantId', 'lastRefresh', 'hash']


In [ ]:
# Identify key columns in raw PA projects file
fips_col       = next((c for c in pa.columns if 'fips' in c.lower()), None)
disaster_col   = next((c for c in pa.columns if 'disaster' in c.lower() and 'number' in c.lower()), 'disasterNumber')
date_col       = next((c for c in pa.columns if 'declarationdate' in c.lower() or 'incidentbegin' in c.lower()), None)
state_col      = next((c for c in pa.columns if 'stateabb' in c.lower()), None)
obligation_col = next((c for c in pa.columns if 'federal' in c.lower() and 'share' in c.lower()), None)

print(f'FIPS col        : {fips_col}')
print(f'Disaster col    : {disaster_col}')
print(f'Date col        : {date_col}')
print(f'State col       : {state_col}')
print(f'Obligation col  : {obligation_col}')

# Build FIPS from state + county codes if no direct FIPS column
if fips_col is None:
    state_num_col  = next((c for c in pa.columns if 'statenumber' in c.lower() or 'statefips' in c.lower()), None)
    county_num_col = next((c for c in pa.columns if 'countycode' in c.lower() or 'countyfips' in c.lower()), None)
    if state_num_col and county_num_col:
        pa['fips'] = pa[state_num_col].astype(str).str.zfill(2) + pa[county_num_col].astype(str).str.zfill(3)
        fips_col = 'fips'
        print(f'Built FIPS from {state_num_col} + {county_num_col}')

# Parse declaration date
if date_col:
    pa[date_col] = pd.to_datetime(pa[date_col], errors='coerce', utc=True).dt.tz_localize(None)

# Build county × disaster matrix — one row per unique county-disaster pair
# NOTE: obligation is NOT included here; it is aggregated separately below
#       to correctly sum all project-level obligations per county-disaster
keep   = [c for c in [fips_col, disaster_col, date_col, state_col] if c]
matrix = pa[keep].drop_duplicates(subset=[fips_col, disaster_col]).copy()
matrix = matrix.rename(columns={
    fips_col:     'fips',
    disaster_col: 'disasterNumber',
    date_col:     'declarationDate',
    state_col:    'stateAbbreviation',
})
matrix['fips'] = matrix['fips'].astype(str).str.zfill(5)

print(f'\nCounty-disaster matrix: {len(matrix):,} rows')
print(f'Unique counties       : {matrix["fips"].nunique():,}')
print(f'Unique disasters      : {matrix["disasterNumber"].nunique():,}')

In [8]:
# Compute prior_disasters_5yr per county
# For each county × disaster row, count how many distinct disasters included
# this county in the 5 years BEFORE the declaration date

if 'declarationDate' in matrix.columns and matrix['declarationDate'].notna().sum() > 0:
    matrix = matrix.sort_values('declarationDate').reset_index(drop=True)
    prior_counts = []
    for _, row in matrix.iterrows():
        if pd.isna(row['declarationDate']):
            prior_counts.append(0)
            continue
        cutoff = row['declarationDate'] - pd.DateOffset(years=5)
        count = matrix[
            (matrix['fips'] == row['fips']) &
            (matrix['declarationDate'] >= cutoff) &
            (matrix['declarationDate'] < row['declarationDate'])
        ]['disasterNumber'].nunique()
        prior_counts.append(count)
    matrix['prior_disasters_5yr'] = prior_counts
    print(f'Prior disasters computed. Mean: {matrix["prior_disasters_5yr"].mean():.2f}')
else:
    matrix['prior_disasters_5yr'] = 0
    print('No date column available — prior_disasters_5yr set to 0')

Prior disasters computed. Mean: 2.19


In [10]:
# Aggregate PA obligation per county-disaster (sum of all projects)
# obligation_col holds the original column name in pa (e.g. 'federalShareObligated')
if obligation_col:
    obligation = (
        pa.assign(fips=pa[fips_col].astype(str).str.zfill(5))
          .groupby(['fips', disaster_col])[obligation_col]
          .sum()
          .reset_index()
          .rename(columns={disaster_col: 'disasterNumber',
                            obligation_col: 'county_obligation'})
    )
    matrix = matrix.merge(obligation, on=['fips', 'disasterNumber'], how='left')
    print(f'Obligation merged. Non-null: {matrix["county_obligation"].notna().sum():,}')

# Join SVI
merged = matrix.merge(
    svi[['FIPS', 'STATE', 'COUNTY', 'svi_score', 'population', 'area_sqmi', 'pop_density']],
    left_on='fips', right_on='FIPS', how='left'
)

svi_match = merged['svi_score'].notna().mean()
print(f'SVI match rate    : {svi_match:.1%}')
print(f'Final merged shape: {merged.shape}')
print(merged[['fips', 'disasterNumber', 'svi_score', 'pop_density', 'prior_disasters_5yr']].head())

Obligation merged. Non-null: 29,329
SVI match rate    : 95.3%
Final merged shape: (29329, 14)
    fips  disasterNumber  svi_score  pop_density  prior_disasters_5yr
0  48465            1239     0.9678    15.590120                    0
1  48029            1239     0.8258  1552.849016                    0
2  48000            1239        NaN          NaN                    0
3  48453            1239     0.3812  1213.054744                    0
4  48267            1239     0.5885     3.523625                    0


In [11]:
# 1. Filter and view the first 5 rows where the county had at least 1 prior disaster
print("\n--- Counties with Prior Disasters ---")
mask_prior = merged['prior_disasters_5yr'] > 0
print(merged[mask_prior][['fips', 'disasterNumber', 'STATE', 'COUNTY', 'prior_disasters_5yr']].head())

# 2. See the full distribution (How many counties had 1, 2, 3+ prior disasters?)
print("\n--- Value Counts of Prior Disasters ---")
print(merged['prior_disasters_5yr'].value_counts().sort_index())


--- Counties with Prior Disasters ---
      fips  disasterNumber      STATE    COUNTY  prior_disasters_5yr
23   48000            1257        NaN       NaN                    1
31   48453            1257      TEXAS    Travis                    1
34   48029            1257      TEXAS     Bexar                    1
224  47051            1262  TENNESSEE  Franklin                    1
226  47003            1262  TENNESSEE   Bedford                    1

--- Value Counts of Prior Disasters ---
prior_disasters_5yr
0     6292
1     7245
2     5913
3     3972
4     2429
5     1299
6      771
7      521
8      294
9      201
10     130
11      83
12      62
13      41
14      24
15      26
16       9
17       7
18       2
19       1
20       3
21       2
22       1
23       1
Name: count, dtype: int64


In [13]:
# Drop statewide summaries (FIPS ending in '000') to keep data strictly at the county level
merged = merged[~merged['fips'].str.endswith('000')].copy()

# Double check how many NaNs are left in SVI
print(f"Remaining NaNs in SVI Score: {merged['svi_score'].isna().sum()}")

Remaining NaNs in SVI Score: 717


In [ ]:
REQUIRED_COLS = ['fips', 'disasterNumber', 'svi_score', 'pop_density',
                  'prior_disasters_5yr', 'county_obligation', 'stateAbbreviation']
missing = [c for c in REQUIRED_COLS if c not in merged.columns]
if missing:
    raise ValueError(f'Missing required columns before save: {missing}')

# Drop redundant columns before saving
DROP_COLS = [c for c in ['FIPS', 'population', 'area_sqmi'] if c in merged.columns]
merged = merged.drop(columns=DROP_COLS)

merged.to_csv(PROCESSED + 'county_disaster_matrix.csv', index=False)
print(f'Saved: county_disaster_matrix.csv ({len(merged):,} rows)')
print(f'\nColumns saved: {list(merged.columns)}')
print(f'\nFeature completeness:')
for col in ['svi_score', 'pop_density', 'prior_disasters_5yr', 'county_obligation']:
    n = merged[col].notna().sum()
    print(f'  {col:<25}: {n:,} / {len(merged):,} ({n/len(merged):.1%})')